In [1]:
import os
os.chdir("../../")#与easymlops同级目录

### 拆分数据

In [2]:
import pandas as pd
data=pd.read_csv("./data/demo.csv")
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
x_train=data[:500]
x_test=data[500:]
y_train=x_train["Survived"]
y_test=x_test["Survived"]
del x_train["Survived"]
del x_test["Survived"]

### pipe构建

In [4]:
from easymlops import TablePipeLine
from easymlops.table.preprocessing import *
from easymlops.table.encoding import *
from easymlops.table.classification import *

In [5]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(OneHotEncoding(cols=["Pclass", "Sex"], drop_col=False)) \
  .pipe(WOEEncoding(cols=["Ticket", "Embarked", "Cabin", "Sex", "Pclass"], y=y_train)) \
  .pipe(LabelEncoding(cols=["Name"]))\
  .pipe(LGBMClassification(y=y_train.map({0:"死亡",1:"存活"}),native_init_params={"max_depth":2},native_fit_params={"num_boost_round":128},drop_input_data=False))

x_test_new=table.fit(x_train).transform(x_test)
x_test_new.head(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Pclass_3,Pclass_1,Pclass_2,Sex_male,Sex_female,死亡,存活
500,501,0.482439,0,1.111379,17.0,0,0,0.0,8.664062,0.299607,0.224849,1,0,0,1,0,0.923326,0.076674
501,502,0.482439,0,-1.569990,21.0,0,0,0.0,7.750000,0.299607,-0.508609,1,0,0,0,1,0.373652,0.626348
502,503,0.482439,0,-1.569990,0.0,0,0,0.0,7.628906,0.299607,-0.508609,1,0,0,0,1,0.378380,0.621620
503,504,0.482439,0,-1.569990,37.0,0,0,0.0,9.585938,0.299607,0.224849,1,0,0,0,1,0.670166,0.329834
504,505,-0.741789,0,-1.569990,16.0,0,0,0.0,86.500000,0.000000,0.224849,0,1,0,0,1,0.068470,0.931530


In [6]:
# table=TablePipeLine()
# table.pipe(FixInput())\
#   .pipe(FillNa())\
#   .pipe(OneHotEncoding(cols=["Pclass", "Sex"], drop_col=False)) \
#   .pipe(WOEEncoding(cols=["Ticket", "Embarked", "Cabin", "Sex", "Pclass"], y=y_train)) \
#   .pipe(LabelEncoding(cols=["Name"]))\
#   .pipe(LGBMClassification(y=y_train.map({0:"死亡",1:"存活"}),native_init_params={"max_depth":2},native_fit_params={"num_boost_round":128}))

# x_test_new=table.fit(x_train).transform(x_test)
# x_test_new.head(5)

In [7]:
#获取sabass特征重要性
table[-1].get_contrib(table.transform_single(x_test.to_dict("record")[0],run_to_layer=-2))

{0: {'Sex': 0.5461501854144144,
  'Cabin': 0.1270063345547848,
  'Age': -0.05115085845832483,
  'Fare': 0.1265214828241528,
  'SibSp': -0.0441638895513395,
  'Ticket': 0.005658456561668252,
  'PassengerId': 0.2263838236137358,
  'Pclass_2': 0.016025418181148575},
 1: {'Sex': -0.5461501854144144,
  'Cabin': -0.1270063345547848,
  'Age': 0.05115085845832483,
  'Fare': -0.1265214828241528,
  'SibSp': 0.0441638895513395,
  'Ticket': -0.005658456561668252,
  'PassengerId': -0.2263838236137358,
  'Pclass_2': -0.016025418181148575}}

### 性能、一致性测试

In [ ]:
table.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.43ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.1ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.OneHotEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.WOEEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.LabelEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.classification.LGBMClassification'>) module check [transform] complete,speed:[4.69ms]/it,cpu:[100%],memory:[0K]

#############################

### 持久化测试

In [9]:
table.save("ml.pkl")

In [10]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(OneHotEncoding(cols=["Pclass", "Sex"], drop_col=False)) \
  .pipe(WOEEncoding(cols=["Ticket", "Embarked", "Cabin", "Sex", "Pclass"])) \
  .pipe(LabelEncoding(cols=["Name"]))\
  .pipe(LGBMClassification(native_init_params={"max_depth":2},native_fit_params={"num_boost_round":128}))

table.load("ml.pkl")

In [11]:
table.auto_test(x_test[:10])


###################################################################
 1.一致性测试和性能测试:check_transform_function                      
###################################################################
(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.1ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.OneHotEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.WOEEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.LabelEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.classification.LGBMClassification'>) module check [transform] complete,speed:[4.69ms]/it,cpu:[84%],memory:[0K]

###############################

In [12]:
table.transform(x_test[:5])

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Pclass_3,Pclass_1,Pclass_2,Sex_male,Sex_female,死亡,存活
500,501,0.482439,0,1.111379,17.0,0,0,0.0,8.664062,0.299607,0.224849,1,0,0,1,0,0.923326,0.076674
501,502,0.482439,0,-1.569990,21.0,0,0,0.0,7.750000,0.299607,-0.508609,1,0,0,0,1,0.373652,0.626348
502,503,0.482439,0,-1.569990,0.0,0,0,0.0,7.628906,0.299607,-0.508609,1,0,0,0,1,0.378380,0.621620
503,504,0.482439,0,-1.569990,37.0,0,0,0.0,9.585938,0.299607,0.224849,1,0,0,0,1,0.670166,0.329834
504,505,-0.741789,0,-1.569990,16.0,0,0,0.0,86.500000,0.000000,0.224849,0,1,0,0,1,0.068470,0.931530


In [13]:
x_test_new.head(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Pclass_3,Pclass_1,Pclass_2,Sex_male,Sex_female,死亡,存活
500,501,0.482439,0,1.111379,17.0,0,0,0.0,8.664062,0.299607,0.224849,1,0,0,1,0,0.923326,0.076674
501,502,0.482439,0,-1.569990,21.0,0,0,0.0,7.750000,0.299607,-0.508609,1,0,0,0,1,0.373652,0.626348
502,503,0.482439,0,-1.569990,0.0,0,0,0.0,7.628906,0.299607,-0.508609,1,0,0,0,1,0.378380,0.621620
503,504,0.482439,0,-1.569990,37.0,0,0,0.0,9.585938,0.299607,0.224849,1,0,0,0,1,0.670166,0.329834
504,505,-0.741789,0,-1.569990,16.0,0,0,0.0,86.500000,0.000000,0.224849,0,1,0,0,1,0.068470,0.931530
